# Part 3 — Multi-Agent Supervisor

Revenue Agent + Expenditure Agent (`agents.py`) under a LangGraph supervisor (`part3_supervisor.py`,
`langgraph_supervisor.create_supervisor`). See `README.md` for the full writeup: architecture,
assumptions, and a real routing-quality finding from development (a demo query initially only
invoked one agent due to accidental context overlap between the two agents' page scoping).

## Sub-agents, verified individually before wiring the supervisor

Isolating failure points: confirm each agent answers correctly on its own before testing the
supervisor's routing on top of them.

In [1]:
from agents import build_revenue_agent, build_expenditure_agent

revenue_agent = build_revenue_agent()
r = revenue_agent.invoke({"messages": [{"role": "user", "content": "What is the largest single source of government revenue in FY2024, and what is its amount?"}]})
print("REVENUE AGENT (standalone):")
print(r["messages"][-1].content)

REVENUE AGENT (standalone):
Based on Table 2.1 in the FY2024 Budget, the largest single source of government revenue in FY2024 is **Corporate Income Tax at $28.03 billion**.

This is followed by Personal Income Tax at $18.07 billion and Goods and Services Tax at $19.39 billion. However, Corporate Income Tax remains the single largest revenue source for FY2024.


In [2]:
expenditure_agent = build_expenditure_agent()
r = expenditure_agent.invoke({"messages": [{"role": "user", "content": "How much is the Future Energy Fund being topped up by, and what will the money be used for?"}]})
print("EXPENDITURE AGENT (standalone):")
print(r["messages"][-1].content)

EXPENDITURE AGENT (standalone):
Based on the expenditure context, the **Future Energy Fund is being topped up by $5.0 billion**.

The money will be used to **invest in critical infrastructure for the energy transition**.

This is an initial injection to establish the new Future Energy Fund as part of Budget 2024's commitment to supporting Singapore's energy transition efforts.


## Supervisor: assignment's exact required query

"What are the key government revenue streams, and how will the Budget for the Future Energy Fund
be supported?" — the trace below shows the supervisor's actual routing decisions, not an assumed
or asserted mechanism: question -> routes to revenue_agent -> tool call -> revenue synthesis ->
back to supervisor -> routes to expenditure_agent -> tool call -> expenditure synthesis -> back to
supervisor -> final comprehensive synthesis.

In [3]:
from part3_supervisor import build_supervisor, run_query, print_trace, DEMO_QUERIES

app = build_supervisor()
q1_label = "Q1 (assignment's exact query, dual-agent)"
answer, trace = run_query(app, DEMO_QUERIES[q1_label])
print("TRACE:")
print_trace(trace)
print("\nAGENTS INVOKED:", sorted(set(t["actor"] for t in trace) - {"user", "supervisor"}))
print("\nFINAL ANSWER:\n", answer)

2026-09-12 21:57:24,729 INFO part3_run_query['What are the key government revenue stre']: start


2026-09-12 21:58:02,488 INFO part3_run_query['What are the key government revenue stre']: done in 37.76s


TRACE:
[1] HumanMessage   actor=user                     What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?
[2] AIMessage      actor=supervisor               thinking: {'signature': 'Ep0DCpABCBEYAipAPvKPQYC+dtcshsT17cz5vk81FOPdRAXvuzVNJGtHPp1k6w0GGbcnTR38ST7aZ20hxPJCfSafqQfwhfJn3QzRfDIPY | tool_use: {'id': 'toolu_01TAx9YY6214dqK4Fb39UaDw', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'transfer_to_revenue_agent',
[3] ToolMessage    actor=transfer_to_revenue_agent Successfully transferred to revenue_agent
[4] AIMessage      actor=revenue_agent            thinking: {'signature': 'ErEDCpABCBEYAipA/2d5VfjQmEHU3ITDnzQaVv0B1y9DnlFPyZW8dh+qm1ZglDi1G5mR+hCYHRlYQ0Da5j5qkDKon9UWGOYL/RYZNzIPY | tool_use: {'id': 'toolu_011oHTromdeQbyxsJ4QNEXt5', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'revenue_context', 'type': '
[5] ToolMessage    actor=revenue_context          --- PAGE 5 ---
MINISTRY OF FINANCE 
 
5 
 
01 Update on Financ

## Verification against ground truth

- Future Energy Fund must be stated as **$5.0 billion**, with the "critical infrastructure for the
  energy transition" purpose from page 18 — not just a bare number.
- Both `revenue_agent` and `expenditure_agent` must appear in the trace's actor set (verified from
  the trace structure itself, not inferred from the answer sounding plausible).
- Revenue streams named in the answer must match the real 12-item Operating Revenue tax list from
  Part 1 (no invented categories).

In [4]:
actors = set(t["actor"] for t in trace)
checks = {
    "Future Energy Fund states $5.0 billion": "5.0 billion" in answer or "$5.0" in answer,
    "Future Energy Fund purpose (energy transition) present": "energy transition" in answer.lower(),
    "revenue_agent invoked (from trace)": "revenue_agent" in actors,
    "expenditure_agent invoked (from trace)": "expenditure_agent" in actors,
}
for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {label}")

PASS  Future Energy Fund states $5.0 billion
PASS  Future Energy Fund purpose (energy transition) present
PASS  revenue_agent invoked (from trace)
PASS  expenditure_agent invoked (from trace)


## Various queries demonstrating collaborative routing

Four queries designed to prove the supervisor genuinely **routes** (not reflexively calling both
agents every time) as well as genuinely **collaborates** when a query needs both:
1. The assignment's exact query (dual-agent) — run above.
2. Revenue-only query — should invoke only `revenue_agent`.
3. Expenditure-only query — should invoke only `expenditure_agent`.
4. A second, differently-phrased dual-agent query — should invoke both again, confirming query 1
   wasn't a one-off fluke.

In [5]:
import json

all_results = {q1_label: {"query": DEMO_QUERIES[q1_label], "trace": trace, "final_answer": answer}}

for label, query in DEMO_QUERIES.items():
    if label == q1_label:
        continue
    print(f"\n{'=' * 80}\n{label}\nQuery: {query}\n{'=' * 80}")
    a, t = run_query(app, query)
    print("\nAGENTS INVOKED:", sorted(set(x["actor"] for x in t) - {"user", "supervisor"}))
    print("\nFINAL ANSWER:\n", a)
    all_results[label] = {"query": query, "trace": t, "final_answer": a}

with open("trace.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)
print("\nWrote trace.json with all 4 demo queries.")

2026-09-12 21:58:02,503 INFO part3_run_query['What is the largest single source of gov']: start



Q2 (revenue-only, tests selective routing)
Query: What is the largest single source of government revenue?


2026-09-12 21:58:16,241 INFO part3_run_query['What is the largest single source of gov']: done in 13.74s


2026-09-12 21:58:16,242 INFO part3_run_query['How much is being spent on the GST Vouch']: start



AGENTS INVOKED: ['revenue_agent', 'revenue_context', 'transfer_back_to_supervisor', 'transfer_to_revenue_agent']

FINAL ANSWER:
 Corporate Income Tax is the largest single source of government revenue.

- **FY2023 (Revised):** $28.4 billion, or **27.2%** of total Operating Revenue — the largest share among all revenue lines (per Chart 1.1 breakdown).
- **FY2024 (Estimated):** $28.03 billion, a slight decrease of $0.35 billion (‑1.2%) from FY2023, but still the top revenue contributor.

For comparison, other major sources trail behind:
- **Personal Income Tax:** $17.53B (FY2023) → $18.07B (FY2024 est.), ~16.8% share
- **Goods and Services Tax:** $16.36B (FY2023) → $19.39B (FY2024 est.), ~15.7% share, growing fastest at +18.5% y/y

While GST is expanding quickly and narrowing the gap, Corporate Income Tax remains the single largest revenue source for the Singapore government in both FY2023 and FY2024.

Q3 (expenditure-only, tests selective routing)
Query: How much is being spent on the 

2026-09-12 21:58:25,213 INFO part3_run_query['How much is being spent on the GST Vouch']: done in 8.97s


2026-09-12 21:58:25,214 INFO part3_run_query['What is the largest source of tax revenu']: start



AGENTS INVOKED: ['expenditure_agent', 'expenditure_context', 'transfer_back_to_supervisor', 'transfer_to_expenditure_agent']

FINAL ANSWER:
 **Summary: GST Voucher Fund Top-up**

- **Amount:** $6.0 billion (the largest single Fund top-up in Budget 2024, out of a total $20.4 billion committed to all Endowment and Trust Fund top-ups)
- **Reason:** The top-up is meant to meet the steady-state cashflow needs of the enhanced, permanent GST Voucher scheme — essentially ensuring the scheme has enough funding set aside to sustainably support its enhanced payouts to Singaporeans over the long term, without needing recurring fresh budget injections each year.

This item falls purely under government expenditure (a Fund top-up), with no revenue-side component to address.

Q4 (second dual-agent query, different phrasing)
Query: What is the largest source of tax revenue, and separately, what specific purpose will the Future Energy Fund infrastructure spending serve according to the budget document

2026-09-12 21:58:49,288 INFO part3_run_query['What is the largest source of tax revenu']: done in 24.07s



AGENTS INVOKED: ['expenditure_agent', 'expenditure_context', 'revenue_agent', 'revenue_context', 'transfer_back_to_supervisor', 'transfer_to_expenditure_agent', 'transfer_to_revenue_agent']

FINAL ANSWER:
 ## Comprehensive Answer

**1. Largest source of tax revenue:**
Corporate Income Tax is the largest source of tax revenue in the government budget. According to the budget document, it accounted for **27.2% of Operating Revenue** in Revised FY2023, with collections of **$28.38 billion**, and is estimated at **$28.03 billion for FY2024** — remaining the top revenue source, ahead of Goods and Services Tax ($19.39 billion estimated FY2024) and Personal Income Tax ($18.07 billion estimated FY2024).

**2. Purpose of the Future Energy Fund infrastructure spending:**
The budget document states that the Government will **establish the Future Energy Fund with an initial injection of $5.0 billion** in Budget 2024. Its specific purpose is **to invest in critical infrastructure supporting Singap

## Routing-pattern summary

Expected pattern: dual / revenue-only / expenditure-only / dual. Confirms the supervisor's
routing decisions are genuine and query-dependent, not hardcoded or reflexive.

In [6]:
for label, result in all_results.items():
    agents_used = sorted(set(t["actor"] for t in result["trace"]) & {"revenue_agent", "expenditure_agent"})
    print(f"{label}: {agents_used}")

Q1 (assignment's exact query, dual-agent): ['expenditure_agent', 'revenue_agent']
Q2 (revenue-only, tests selective routing): ['revenue_agent']
Q3 (expenditure-only, tests selective routing): ['expenditure_agent']
Q4 (second dual-agent query, different phrasing): ['expenditure_agent', 'revenue_agent']
